## In-Class Activity 04/21

**#1.** Build at least two models. You may choose any two models, but they should differ in a meaningful way, either in model type, feature representation, or tuning choices. This could include using a non-tree model (e.g., logistic regression, SVM) to compare how different models respond to feature engineering.

In [14]:
# importing libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV


In [2]:
# importing the data
adult = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/April 20-26/adult.csv')
adult.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
# replace ? placeholders with missing values
adult = adult.replace('?', np.nan)

# convert target variable to binary
adult['income'] = (adult['income'] == '>50K').astype(int)

# split features and target
X = adult.drop(columns=['income'])
y = adult['income']

# identify numeric vs categorical columns for the pipeline
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# fixed split so every model sees the same train/test rows
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print('Train rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])
print('Numeric columns:', len(numeric_features))
print('Categorical columns:', len(categorical_features))


Train rows: 39073
Test rows: 9769
Numeric columns: 6
Categorical columns: 8


In [4]:
# numeric pipeline: impute then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# categorical pipeline: impute then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# combine both pipelines into one reusable preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [7]:
# 5-fold CV for consistent model comparison
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# model 1: linear baseline (L2 logistic regression, no L1 penalty)
log_reg = LogisticRegression(max_iter=2000, solver='liblinear')

# model 2: tree-based baseline
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# wrap each model with the same preprocessing pipeline
pipelines = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('model', log_reg)]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('model', rf)])
}

results = []

for name, model_pipeline in pipelines.items():
    # cross-validation performance on training set
    cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=kf, scoring='accuracy')

    # fit on full training split and evaluate on holdout test split
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)

    print(f'\n{name}')
    print('-' * len(name))
    print(f'CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print(f'Test Accuracy: {test_acc:.4f}')
    print('Classification Report:')
    print(classification_report(y_test, y_pred))

    results.append({
        'model': name,
        'cv_accuracy_mean': cv_scores.mean(),
        'cv_accuracy_std': cv_scores.std(),
        'test_accuracy': test_acc
    })

# side-by-side baseline comparison
comparison_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False).reset_index(drop=True)
comparison_df


/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Logistic Regression
-------------------
CV Accuracy: 0.8514 +/- 0.0053
Test Accuracy: 0.8524
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.74      0.59      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.76      0.78      9769
weighted avg       0.85      0.85      0.85      9769



/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Random Forest
-------------
CV Accuracy: 0.8526 +/- 0.0045
Test Accuracy: 0.8583
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      7431
           1       0.74      0.62      0.68      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.78      0.79      9769
weighted avg       0.85      0.86      0.85      9769



,model,cv_accuracy_mean,cv_accuracy_std,test_accuracy
0,Random Forest,0.852609,0.004509,0.858327
1,Logistic Regression,0.851380,0.005265,0.852390


**#2.** Create useful new features. These should not all be variations of the same idea. Instead, use a mix of approaches such as interactions, grouped categories, or transformations. The goal is to explore different ways of representing data, not just repeating one pattern. Note that model performance is not the only way to define usefulness. Features that improve interpretability or reveal structure in the data are also valuable.


In [10]:
##interactions

df = adult.copy()

# education intensity of work
df["edu_x_hours"] = df["educational-num"] * df["hours-per-week"]

# life stage + workload
df["age_x_hours"] = df["age"] * df["hours-per-week"]

# education + occupation
df["edu_x_occupation"] = df["educational-num"] * df["occupation"].astype('category').cat.codes


In [11]:
##grouped categories

#turn country into region
def country_to_region(c):
    if c == "United-States":
        return "US"
    if c in {"Canada", "Mexico"}:
        return "NorthAmerica_nonUS"
    if c in {"India", "China", "Japan", "Philippines", "Vietnam"}:
        return "Asia"
    if c in {"England", "Germany", "Italy", "France", "Poland"}:
        return "Europe"
    if c in {"Puerto-Rico", "Cuba", "Jamaica", "Dominican-Republic", "El-Salvador"}:
        return "LatinAmerica_Caribbean"
    return "Other"
df["native_region"] = df["native-country"].fillna("Unknown").apply(country_to_region)



In [12]:
##transformations

# reduce heavy skew
df["log_capital_gain"] = np.log1p(df["capital-gain"])
df["log_capital_loss"] = np.log1p(df["capital-loss"])

# combine related monetary signals
df["net_capital"] = df["capital-gain"] - df["capital-loss"]

#workload buckets
df["hours_band"] = pd.cut(
    df["hours-per-week"],
    bins=[0, 34, 40, 50, 100],
    labels=["PartTime", "FullTime", "Heavy", "VeryHeavy"],
    include_lowest=True)

**#3.** Tune your models beyond their default settings. There is no fixed number of parameters you must tune, but you should make a reasonable effort to improve performance and demonstrate that your tuning choices had an effect.

In [16]:
# tune Logistic Regression 

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

log_param_grid = {
    "model__C": [0.1, 0.5, 1.0, 2.0, 5.0],      # regularization strength
    "model__solver": ["liblinear"],             # stable/simple for this class setup
    "model__class_weight": [None, "balanced"]   # try imbalance handling
}

log_search = GridSearchCV(
    estimator=log_pipe,
    param_grid=log_param_grid,
    cv=kf,
    scoring="accuracy"
)

log_search.fit(X_train, y_train)

best_log = log_search.best_estimator_
log_pred = best_log.predict(X_test)
log_test_acc = accuracy_score(y_test, log_pred)

print("Best Logistic Params:", log_search.best_params_)
print(f"Best Logistic CV Accuracy: {log_search.best_score_:.4f}")
print(f"Tuned Logistic Test Accuracy: {log_test_acc:.4f}")
print(classification_report(y_test, log_pred))


/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.1

Best Logistic Params: {'model__C': 0.5, 'model__class_weight': None, 'model__solver': 'liblinear'}
Best Logistic CV Accuracy: 0.8514
Tuned Logistic Test Accuracy: 0.8524
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.74      0.59      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.76      0.78      9769
weighted avg       0.85      0.85      0.85      9769



In [18]:
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# Keep it small and meaningful
rf_param_grid_fast = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 3]
}

kf_fast = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_param_grid_fast,
    cv=kf_fast,
    scoring="accuracy",
    verbose=1
    # n_jobs=-1  # optional: use all cores; if it errors on your machine, remove this
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
rf_pred = best_rf.predict(X_test)
rf_test_acc = accuracy_score(y_test, rf_pred)

print("Best RF Params:", rf_search.best_params_)
print(f"Best RF CV Accuracy: {rf_search.best_score_:.4f}")
print(f"Tuned RF Test Accuracy: {rf_test_acc:.4f}")
print(classification_report(y_test, rf_pred))

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.1

Best RF Params: {'model__max_depth': None, 'model__min_samples_leaf': 3, 'model__n_estimators': 250}
Best RF CV Accuracy: 0.8628
Tuned RF Test Accuracy: 0.8662
              precision    recall  f1-score   support

           0       0.88      0.95      0.92      7431
           1       0.79      0.60      0.68      2338

    accuracy                           0.87      9769
   macro avg       0.84      0.78      0.80      9769
weighted avg       0.86      0.87      0.86      9769



In [19]:
# Compare baseline vs tuned performance for each model

comparison_simple = pd.DataFrame([
    {"model": "Logistic Regression", "baseline_test_accuracy": 0.8524, "tuned_test_accuracy": log_test_acc},
    {"model": "Random Forest", "baseline_test_accuracy": 0.8583, "tuned_test_accuracy": rf_test_acc}
])

comparison_simple["accuracy_gain"] = (
    comparison_simple["tuned_test_accuracy"] - comparison_simple["baseline_test_accuracy"]
)

comparison_simple = comparison_simple.sort_values("tuned_test_accuracy", ascending=False).reset_index(drop=True)
comparison_simple


,model,baseline_test_accuracy,tuned_test_accuracy,accuracy_gain
0,Random Forest,0.8583,0.866209,0.007909
1,Logistic Regression,0.8524,0.852390,-0.000010


There was a strong accuracy gain for the Random Forest model (+0.007909), but there was effectively no difference in tuned vs untuned Logistic Regression (-.000010).

**#4.** Combine your models using probability averaging. You may use equal weights or choose your own weighting scheme. The purpose is to explore whether combining models leads to an improvement over individual models.

In [21]:
log_probs = best_log.predict_proba(X_test)[:, 1]
rf_probs = best_rf.predict_proba(X_test)[:, 1]

avg_probs_equal = (log_probs + rf_probs) / 2
y_pred_equal = (avg_probs_equal >= 0.5).astype(int)
equal_acc = accuracy_score(y_test, y_pred_equal)

print("Equal-Weight Ensemble Accuracy:", round(equal_acc, 4))
print(classification_report(y_test, y_pred_equal))



Equal-Weight Ensemble Accuracy: 0.8627
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.77      0.60      0.68      2338

    accuracy                           0.86      9769
   macro avg       0.83      0.77      0.80      9769
weighted avg       0.86      0.86      0.86      9769



**#5.** Evaluate your results. Briefly describe which features seemed most useful, which model performed best, and whether the ensemble improved performance. Also comment on whether different models responded differently to your engineered features. Summarize how this will fit into your feature engineering and model building workflow.

In [23]:
# feature names after preprocessing
rf_feature_names = best_rf.named_steps["preprocessor"].get_feature_names_out()
log_feature_names = best_log.named_steps["preprocessor"].get_feature_names_out()

# Random Forest importances
rf_importance = pd.Series(
    best_rf.named_steps["model"].feature_importances_,
    index=rf_feature_names
).sort_values(ascending=False)

# Logistic coefficients (absolute magnitude = strength)
log_coef = pd.Series(
    best_log.named_steps["model"].coef_[0],
    index=log_feature_names
)
log_strength = log_coef.reindex(log_coef.abs().sort_values(ascending=False).index)

print("Top 15 Random Forest features:")
display(rf_importance.head(15).to_frame("importance"))

print("\nTop 15 Logistic features (with sign):")
display(log_strength.head(15).to_frame("coefficient"))


Top 15 Random Forest features:


,importance
cat__marital-status_Married-civ-spouse,0.171311
num__capital-gain,0.141919
num__educational-num,0.110858
num__age,0.088723
cat__marital-status_Never-married,0.059916
num__hours-per-week,0.057439
num__capital-loss,0.042343
num__fnlwgt,0.038671
cat__occupation_Exec-managerial,0.031404
cat__relationship_Not-in-family,0.029973



Top 15 Logistic features (with sign):


,coefficient
num__capital-gain,2.252414
cat__marital-status_Married-civ-spouse,1.909967
cat__marital-status_Married-AF-spouse,1.321290
cat__native-country_Columbia,-1.203388
cat__occupation_Priv-house-serv,-1.122957
cat__relationship_Wife,1.088077
cat__workclass_Self-emp-not-inc,-1.078852
cat__occupation_Farming-fishing,-1.014234
cat__occupation_Other-service,-0.925226
cat__native-country_South,-0.910545


- I evaluated two tuned models on the same train/test split: Logistic Regression and Random Forest, and the best individual model was tuned Random Forest with test accuracy 0.8662

    - Tuned Logistic Regression achieved test accuracy 0.8524.

    - The equal-weight probability ensemble achieved 0.8627.

- The ensemble improved over Logistic Regression, but did not beat the tuned Random Forest.

- The most useful features in Random Forest were `marital-status_Married-civ-spouse`, `capital-gain`, `educational-num`, `age`, and `hours-per-week`

- Logistic Regression highlighted similar core signals (`capital-gain`, `educational-num`, `marital-status` terms), but also gave large coefficients to some sparse category levels like specific country/occupation dummies

- The models responded differently to the engineered feature space: Random Forest benefited more from tuning and captured nonlinear interactions better, while Logistic Regression was more stable and interpretable but showed limited performance gains. This supports using both model families during development: linear models for interpretation and tree models for stronger predictive performance

For future workflows I will probably start with baseline models, add diverse feature engineering (interactions, grouped categories, transformations), tune each model, then test probability averaging to confirm whether blending improves over the best single model

- In this case, blending was useful as a comparison checkpoint, but the tuned Random Forest remained a better choice
